In [10]:
import copy,random,re,time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sacrebleu
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence
from torch.utils.data import DataLoader

In [11]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
P_TR="../en_fr_train.csv"
P_VA="../en_fr_val.csv"
P_TE="../en_fr_test.csv"
N=200000
MAXLEN=40
DMAX=100
MINF=3
BS=128
EPOCHS=15
PAT=3
LR=1e-3
D=128
H=256
L=2
DR=0.3
PAD,SOS,EOS,UNK=0,1,2,3
DEV="cuda" if torch.cuda.is_available() else "cpu"

In [9]:
tok=lambda s:" ".join(re.findall(r"\w+|[^\w\s]",str(s).lower()))
def ld(p):
    d=pd.read_csv(p)[["en","fr"]].dropna()
    d["en"]=d["en"].map(tok)
    d["fr"]=d["fr"].map(tok)
    return d[(d.en!="")&(d.fr!="")]
def cap(d):
    a=d.en.str.split().str.len()
    b=d.fr.str.split().str.len()
    return d[(a<=MAXLEN)&(b<=MAXLEN)]
tr,va,te=ld(P_TR),ld(P_VA),ld(P_TE)
tr=cap(tr.drop_duplicates("en")).sample(frac=1,random_state=SEED).head(N)
va=cap(va)
print(len(tr),len(va),len(te))
print(tr.head())

199459 747 8597
                                                       en  \
79032   we walked into the offices of the municipal co...   
165819  so what we ' re doing is placing these electro...   
56876                david pogue : hi babes , it ' s me .   
203313  it sent me over mass . ave . , massachusetts a...   
99555                         we are learning from them .   

                                                       fr  
79032   nous sommes allés voir la municipalité , la po...  
165819  ce qu ' on fait , c ' est qu ' on introduit ce...  
56876          david pogue : salut chérie , c ' est moi .  
203313  il m ' envoyait sur mass . ave . , massachuset...  
99555                            nous apprenons d ' eux .  


In [12]:
def mkv(S):
    c=Counter(w for s in S for w in s.split())
    i2w=["<pad>","<sos>","<eos>","<unk>"]+[w for w,n in c.items() if n>=MINF]
    return {w:i for i,w in enumerate(i2w)},i2w
s2i,i2s=mkv(tr.en)
t2i,i2t=mkv(tr.fr)
print("English vocab",len(i2s),"| French vocab",len(i2t))

English vocab 23471 | French vocab 29218


In [13]:
def pairs(d):
    return [([s2i.get(w,UNK) for w in a.split()],[t2i.get(w,UNK) for w in b.split()]) for a,b in zip(d.en,d.fr)]
def coll(bt):
    s,t=zip(*bt)
    sl=torch.tensor([len(i) for i in s])
    S=pad_sequence([torch.tensor(i) for i in s],batch_first=True)
    X=pad_sequence([torch.tensor([SOS]+i) for i in t],batch_first=True)
    Y=pad_sequence([torch.tensor(i+[EOS]) for i in t],batch_first=True)
    return S,sl,X,Y
trl=DataLoader(pairs(tr),batch_size=BS,shuffle=True,collate_fn=coll)
val=DataLoader(pairs(va),batch_size=BS,collate_fn=coll)
tel=DataLoader(pairs(te),batch_size=BS,collate_fn=coll)

In [14]:
class Seq2Seq(nn.Module):
    def __init__(self,sv,tv):
        super().__init__()
        self.se=nn.Embedding(sv,D,padding_idx=PAD)
        self.te=nn.Embedding(tv,D,padding_idx=PAD)
        self.enc=nn.LSTM(D,H,L,batch_first=True,dropout=DR)
        self.dec=nn.LSTM(D,H,L,batch_first=True,dropout=DR)
        self.out=nn.Linear(H,tv)
        self.dr=nn.Dropout(DR)
    def encode(self,s,sl):
        return self.enc(pack_padded_sequence(self.dr(self.se(s)),sl.cpu(),batch_first=True,enforce_sorted=False))[1]
    def decode(self,x,st):
        o,st=self.dec(self.dr(self.te(x)),st)
        return self.out(self.dr(o)),st
    def forward(self,s,sl,x):
        return self.decode(x,self.encode(s,sl))[0]
torch.manual_seed(SEED)
m=Seq2Seq(len(i2s),len(i2t)).to(DEV)
print("Parameters:",sum(p.numel() for p in m.parameters()))

Parameters: 16096418


In [ ]:
lf=nn.CrossEntropyLoss(ignore_index=PAD)
opt=torch.optim.Adam(m.parameters(),lr=LR)
def ep(l,train):
    m.train(train)
    s=n=0
    with torch.set_grad_enabled(train):
        for S,sl,X,Y in l:
            S,X,Y=S.to(DEV),X.to(DEV),Y.to(DEV)
            o=m(S,sl,X)
            loss=lf(o.reshape(-1,o.size(-1)),Y.reshape(-1))
            if train:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(),1.0)
                opt.step()
            k=(Y!=PAD).sum().item()
            s+=loss.item()*k
            n+=k
    return s/n
hist={"train":[],"val":[]}
bv,st,bad=1e9,None,0
t0=time.time()
for e in range(1,EPOCHS+1):
    hist["train"].append(ep(trl,True))
    hist["val"].append(ep(val,False))
    print(f"{e:02d} train {hist['train'][-1]:.4f} val {hist['val'][-1]:.4f} val ppl {np.exp(hist['val'][-1]):.2f}")
    if hist["val"][-1]<bv:bv,st,bad=hist["val"][-1],copy.deepcopy(m.state_dict()),0
    else:
        bad+=1
        if bad>=PAT:break
print(f"Training time {time.time()-t0:.0f}s")
m.load_state_dict(st)

In [ ]:
plt.plot(hist["train"],label="train")
plt.plot(hist["val"],label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy per token")
plt.legend()
plt.show()

In [ ]:
def trans(l):
    m.eval()
    out=[]
    with torch.no_grad():
        for S,sl,_,_ in l:
            st=m.encode(S.to(DEV),sl)
            x=torch.full((len(S),1),SOS,dtype=torch.long,device=DEV)
            o=[]
            for _ in range(MAXLEN+2):
                y,st=m.decode(x,st)
                x=y[:,-1].argmax(-1,keepdim=True)
                o.append(x)
            for r in torch.cat(o,1).tolist():
                r=r[:r.index(EOS)] if EOS in r else r
                out.append(" ".join(i2t[i] for i in r))
    return out
bl=lambda h,r:sacrebleu.corpus_bleu(list(h),[list(r)],tokenize="none",force=True).score
Hy=trans(tel)
Rf=te.fr.tolist()
print("Test BLEU:",bl(Hy,Rf))

In [ ]:
sl=te.en.str.split().str.len().to_numpy()
nw=lambda x:x.str.split().str.len().sum()
t=pd.DataFrame({"len":pd.cut(sl,[0,5,10,15,20,1e9],labels=["1-5","6-10","11-15","16-20","21+"]),"h":Hy,"r":Rf})
rows=[{"len":k,"n":len(g),"BLEU":bl(g.h,g.r),"len_ratio":nw(g.h)/nw(g.r)} for k,g in t.groupby("len",observed=True)]
lt=pd.DataFrame(rows).set_index("len")
print(lt.round(2))
lt["BLEU"].plot.bar()
plt.ylabel("BLEU")
plt.show()

In [ ]:
t["unk"] = [any(w not in s2i for w in s.split()) for s in te.en]
for k, g in t.groupby("unk"): print("English has unknown word" if k else "No unknown word", len(g),
                                    round(bl(g.h, g.r), 2))
print("Outputs containing <unk>:", np.mean(["<unk>" in h for h in Hy]))
print("Reference words outside the French vocab:", np.mean([w not in t2i for r in Rf for w in r.split()]))
for i in list(np.random.RandomState(SEED).choice(len(te), 5, replace=False)) + list(np.argsort(-sl)[:5]):
    print("SRC:", te.en.iloc[i], "\nREF:", Rf[i], "\nHYP:", Hy[i], "\n")